# Self-Attention

**Companion lesson:** https://ml-viz.vercel.app/courses/transformers/01-self-attention

A from-scratch, runnable implementation of the concepts in the lesson.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Scaled dot-product attention, from scratch

$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$. We implement it, run it on a short sequence, and inspect the attention weights.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-1, -2) / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

class SelfAttention:
    def __init__(self, d_model, d_k, seed=0):
        rng = np.random.RandomState(seed)
        self.Wq = rng.randn(d_model, d_k)*0.3
        self.Wk = rng.randn(d_model, d_k)*0.3
        self.Wv = rng.randn(d_model, d_k)*0.3
    def __call__(self, X, mask=None):
        return attention(X@self.Wq, X@self.Wk, X@self.Wv, mask)

X = np.random.randn(5, 8)          # 5 tokens, model dim 8
sa = SelfAttention(d_model=8, d_k=8)
out, W = sa(X)
print('output shape:', out.shape, '| attention rows sum to 1:', np.allclose(W.sum(1), 1))

## Worked numeric example, by hand (matches the lesson)

Three tokens, $d_k=2$. Each row of $Q,K,V$ is one token's projected vector:

$$Q=\begin{bmatrix}2&0\\0&2\\2&2\end{bmatrix},\quad K=\begin{bmatrix}2&0\\0&2\\1&1\end{bmatrix},\quad V=\begin{bmatrix}1&0\\0&1\\10&10\end{bmatrix}.$$

1. **Scores** $QK^\top=\begin{bmatrix}4&0&2\\0&4&2\\4&4&4\end{bmatrix}$ (entry $(i,j)=\mathbf q_i\cdot\mathbf k_j$).
2. **Scale** by $1/\sqrt2\approx0.707$.
3. **Softmax** each row, e.g. row 1: $e^{2.828},e^{0},e^{1.414}=16.92,1,4.11$, sum $22.03\Rightarrow[0.768,0.045,0.187]$. Row 3's scores are all equal, so it is exactly uniform $[\tfrac13,\tfrac13,\tfrac13]$.
4. **Weighted sum** of value rows.

The cell below recomputes everything in **pure Python (no numpy)** so every number is deterministic and hand-checkable, then asserts the known answers.

In [ ]:
import math  # pure stdlib: deterministic, hand-checkable

def softmax_row(row):
    e = [math.exp(s) for s in row]
    z = sum(e)
    return [x / z for x in e]

Q = [[2, 0], [0, 2], [2, 2]]
K = [[2, 0], [0, 2], [1, 1]]
V = [[1, 0], [0, 1], [10, 10]]
d_k = 2

# Step 1: scores QKt  -> Step 2: scale -> Step 3: softmax -> Step 4: @V
scores  = [[sum(q[d] * k[d] for d in range(d_k)) for k in K] for q in Q]
scaled  = [[s / math.sqrt(d_k) for s in row] for row in scores]
weights = [softmax_row(row) for row in scaled]
out = [[sum(weights[i][j] * V[j][d] for j in range(3)) for d in range(d_k)]
       for i in range(3)]

print("QKt      :", scores)
print("weights  :", [[round(w, 3) for w in r] for r in weights])
print("row sums :", [round(sum(r), 6) for r in weights])
print("output   :", [[round(o, 3) for o in r] for r in out])

# Deterministic verification against the by-hand answers in the lesson
assert scores == [[4, 0, 2], [0, 4, 2], [4, 4, 4]]
assert all(abs(sum(r) - 1.0) < 1e-12 for r in weights)             # softmax rows normalize
assert [round(w, 3) for w in weights[0]] == [0.768, 0.045, 0.187]  # token 1 attends to key 1
assert [round(w, 3) for w in weights[2]] == [0.333, 0.333, 0.333]  # token 3 is uniform
assert [round(o, 3) for o in out[0]] == [2.635, 1.912]
assert [round(o, 3) for o in out[2]] == [3.667, 3.667]             # uniform -> mean of values
print()
print("All hand-checked values verified.")

## Why divide by $\sqrt{d_k}$? — verify the variance

For unit-variance $\mathbf q,\mathbf k$, the dot product $\mathbf q\cdot\mathbf k=\sum_{i=1}^{d_k}q_ik_i$ is a sum of $d_k$ independent terms each with variance $1$, so $\operatorname{Var}(\mathbf q\cdot\mathbf k)=d_k$. Scores therefore scale like $\sqrt{d_k}$ — without correction, large $d_k$ pushes softmax into its saturated, near-zero-gradient regime. Dividing by $\sqrt{d_k}$ restores variance $1$. The cell below confirms $\operatorname{Var}\approx d_k$ empirically using only the stdlib `random` module (deterministic via a fixed seed).

In [ ]:
import random

def dot_product_variance(d_k, trials=20000, seed=1):
    rng = random.Random(seed)
    vals = []
    for _ in range(trials):
        q = [rng.gauss(0, 1) for _ in range(d_k)]
        k = [rng.gauss(0, 1) for _ in range(d_k)]
        vals.append(sum(qi * ki for qi, ki in zip(q, k)))
    mean = sum(vals) / len(vals)
    return sum((v - mean) ** 2 for v in vals) / len(vals)

for d_k in (2, 8, 64):
    var = dot_product_variance(d_k)
    # scaling by 1/sqrt(d_k) divides the variance by d_k -> ~1
    print("d_k=%2d: Var(q.k)=%6.2f  (theory %d)   scaled Var=%.3f"
          % (d_k, var, d_k, var / d_k))

# unscaled variance grows with d_k; the 1/sqrt(d_k) scaling pins it near 1
assert abs(dot_product_variance(8) - 8) < 1.0
print()
print("Variance grows as d_k; the sqrt(d_k) scaling keeps it ~1.")

## Visualizing what attends to what

In [ ]:
plt.imshow(W if False else sa(X)[1], cmap='magma')
plt.colorbar(label='attention weight'); plt.xlabel('key token'); plt.ylabel('query token')
plt.title('Self-attention weights'); plt.show()

## Causal masking for autoregressive models

GPT-style generation forbids attending to the future. A lower-triangular mask sets future scores to $-\infty$ so their softmax weight is 0 — note the upper triangle is blank.

In [ ]:
T = 5
mask = np.tril(np.ones((T, T))).astype(bool)   # True = allowed
_, Wc = sa(X, mask=mask)
plt.imshow(Wc, cmap='magma')
plt.title('Causal attention (upper triangle = 0)'); plt.xlabel('key'); plt.ylabel('query')
plt.colorbar(); plt.show()
print('row 0 attends only to token 0:', np.round(Wc[0], 3))

## Key takeaways

- Attention is `softmax(QKᵀ/√dₖ)·V` — a content-based weighted average of value vectors.
- Every token reaches every other in **one** step, fully in parallel.
- A **causal mask** enables autoregressive generation.
- The √dₖ scaling keeps softmax out of its saturated, low-gradient regime.

## ✏️ Your turn

### Exercise 1 — Scaled dot-product attention

The formula is:

$$\\text{Attention}(Q,K,V) = \\text{softmax}\\!\\left(\\frac{QK^\\top}{\\sqrt{d_k}}\\right)V$$

Implement it from scratch (you may use `numpy`). Three properties to verify:
- each row of the weight matrix sums to 1 (softmax)
- identical queries produce uniform attention over all keys
- output shape matches `V`

In [ ]:
import numpy as np

def scaled_dot_product_attention(Q, K, V):
    """Return (output, weights) where output = softmax(Q K^T / sqrt(d_k)) @ V."""
    # TODO(you): implement the three steps — score, softmax, weighted sum
    ...

# Quick smoke-test (do not edit)
Q = np.array([[1.0, 0.0], [0.0, 1.0]])
K = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])
V = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])

In [ ]:
out, W = scaled_dot_product_attention(Q, K, V)

assert W.shape == (2, 3), "weights must be (num_queries, num_keys)"
assert np.allclose(W.sum(axis=1), 1.0, atol=1e-6), \
    "each row of the weight matrix must sum to 1 (softmax normalisation)"

# uniform query–key alignment → uniform weights
Q_same = np.ones((4, 3))
K_same = np.ones((5, 3))
V_same = np.random.randn(5, 2)
_, W_unif = scaled_dot_product_attention(Q_same, K_same, V_same)
assert np.allclose(W_unif, 1.0 / 5, atol=1e-6), \
    "when all dot-products are equal, softmax must be uniform (1/num_keys)"

assert out.shape == V.shape, "output shape must equal V.shape"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    scores -= scores.max(axis=-1, keepdims=True)   # numerical stability
    weights = np.exp(scores) / np.exp(scores).sum(axis=-1, keepdims=True)
    return weights @ V, weights
```

</details>

### Exercise 2 — Causal masking

Autoregressive models must not attend to future tokens. A lower-triangular boolean mask
sets future scores to $-\\infty$, so their softmax weight is exactly 0.

Build the mask for a sequence of length $T$, apply it, and verify the
upper triangle of the weight matrix is zero.

In [ ]:
def causal_attention(Q, K, V):
    """Scaled dot-product attention with a causal (lower-triangular) mask."""
    T = Q.shape[0]
    # TODO(you): build the boolean mask (True = allowed) and call
    #            scaled_dot_product_attention with scores masked to -1e9 where False
    ...

In [ ]:
T = 5
d_k = 4
rng2 = np.random.RandomState(7)
Q2 = rng2.randn(T, d_k); K2 = rng2.randn(T, d_k); V2 = rng2.randn(T, d_k)
out2, W2 = causal_attention(Q2, K2, V2)

upper = W2[np.triu_indices(T, k=1)]
assert np.allclose(upper, 0.0, atol=1e-6), \
    "upper triangle of the weight matrix must be 0 (future tokens are masked)"
assert np.allclose(W2.sum(axis=1), 1.0, atol=1e-6), \
    "each row still sums to 1 after masking"
assert W2[0, 0] > 0.99, \
    "first token can only attend to itself, so W[0,0] must be ~1"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def causal_attention(Q, K, V):
    T, d_k = Q.shape
    mask = np.tril(np.ones((T, T))).astype(bool)   # True = allowed
    scores = Q @ K.T / np.sqrt(d_k)
    scores = np.where(mask, scores, -1e9)
    scores -= scores.max(axis=-1, keepdims=True)
    weights = np.exp(scores) / np.exp(scores).sum(axis=-1, keepdims=True)
    return weights @ V, weights
```

</details>